<a href="https://colab.research.google.com/github/AngelTroncoso/Alergias/blob/main/01_An%C3%A1lisis_de_Estructuras_PDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Nuevo Punto de Partida para Análisis de Estructuras PDB

Este cuaderno ha sido limpiado para proporcionar un punto de partida funcional para la descarga y análisis de estructuras PDB. Nos centraremos en dos librerías principales:

*   **`requests`**: Para descargar archivos PDB directamente desde la base de datos RCSB PDB.
*   **`Biopython`**: Para parsear y analizar el contenido de los archivos PDB descargados.

**Nota sobre `rcsbsearchapi`**: La librería `rcsbsearchapi` ha presentado problemas de compatibilidad y errores (`AttributeError`, `ImportError`) con las versiones disponibles, lo que impide su uso fiable para consultas avanzadas. Si necesitas realizar búsquedas complejas en la base de datos, se podría explorar la construcción manual de consultas a la API de RCSB PDB usando `requests` o buscar alternativas en el futuro.

### 1. Descargar archivos PDB usando `requests`

Esta función te permite descargar cualquier archivo PDB conociendo su ID. La URL base para la descarga es `https://files.rcsb.org/download/{pdb_id}.pdb`.

In [ ]:
import requests

def download_pdb_file(pdb_id, output_dir='./'):
    """
    Descarga un archivo PDB desde RCSB PDB.

    Args:
        pdb_id (str): El ID de 4 caracteres de la estructura PDB.
        output_dir (str): Directorio donde guardar el archivo descargado.
    """
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    filename = f"{output_dir}{pdb_id}.pdb"

    print(f"Descargando {pdb_id} de {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status() # Lanza una excepción si la solicitud no fue exitosa

    with open(filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Archivo {filename} descargado exitosamente.")

# Ejemplo de uso: Descargamos el archivo 5DM2.pdb
pdb_id_to_download = "5DM2"
download_pdb_file(pdb_id_to_download)

Descargando 5DM2 de https://files.rcsb.org/download/5DM2.pdb...
Archivo ./5DM2.pdb descargado exitosamente.


### 2. Leer y Analizar Archivos PDB con Biopython

Una vez descargado el archivo, Biopython es la herramienta ideal para cargar la estructura y acceder a sus componentes como modelos, cadenas, residuos y átomos.

In [ ]:
# Instalar Biopython si no está instalado
!pip install biopython

In [ ]:
from Bio.PDB import PDBParser

# Crear un objeto PDBParser
parser = PDBParser()

# Ruta al archivo PDB descargado
pdb_file_path = './5DM2.pdb'

# Parsear la estructura
structure = parser.get_structure('5DM2', pdb_file_path)

print(f"ID de la estructura: {structure.get_id()}")
print(f"Número de modelos: {len(structure)}")

for model in structure:
    print(f"  Modelo ID: {model.get_id()}")
    print(f"  Número de cadenas en el modelo {model.get_id()}: {len(model)}")
    for chain in model:
        print(f"    Cadena ID: {chain.get_id()}")
        print(f"    Número de residuos en la cadena {chain.get_id()}: {len(chain)}")
        # Mostrar los primeros 5 residuos como ejemplo para evitar una salida muy larga
        for i, residue in enumerate(chain):
            if i < 5:
                print(f"      Residuo: {residue.get_resname()}, ID: {residue.get_id()[1]}")
            else:
                break
        if len(chain) > 5:
            print("      ...")


ID de la estructura: 5DM2
Número de modelos: 1
  Modelo ID: 0
  Número de cadenas en el modelo 0: 1
    Cadena ID: A
    Número de residuos en la cadena A: 497
      Residuo: ILE, ID: 2
      Residuo: GLN, ID: 3
      Residuo: GLU, ID: 4
      Residuo: LYS, ID: 5
      Residuo: ILE, ID: 6
      ...


### 3. Extraer y Listar Átomos de una Cadena Específica

Una vez que tienes la estructura cargada, puedes navegar a través de sus componentes (modelos, cadenas, residuos) para acceder a los átomos. Este ejemplo muestra cómo listar todos los átomos de una cadena específica (por ejemplo, la cadena 'A').

In [ ]:
from Bio.PDB import PDBParser

# Asegúrate de que el archivo PDB haya sido descargado y Biopython instalado
# (Esto se asume que ya lo ejecutaste en las celdas anteriores)

# Crear un objeto PDBParser
parser = PDBParser()

# Ruta al archivo PDB descargado
pdb_file_path = './5DM2.pdb'

# Parsear la estructura (si no está ya cargada)
structure = parser.get_structure('5DM2', pdb_file_path)

# Especifica el ID del modelo y de la cadena que te interesa
# En este ejemplo, el modelo 0 y la cadena 'A' son los que existen en 5DM2
model_id_to_explore = 0
chain_id_to_explore = 'A'

print(f"Listando átomos de la Cadena '{chain_id_to_explore}' en el Modelo {model_id_to_explore}...")

atom_count = 0
for model in structure:
    if model.get_id() == model_id_to_explore:
        for chain in model:
            if chain.get_id() == chain_id_to_explore:
                print(f"  Cadena encontrada: {chain.get_id()}")
                for residue in chain:
                    # Imprimir información del residuo para contexto
                    # print(f"    Residuo: {residue.get_resname()} (ID: {residue.get_id()[1]}) -")
                    for atom in residue:
                        print(f"      Átomo: {atom.get_id()}, Elemento: {atom.element}, Coordenadas: {atom.get_coord()}")
                        atom_count += 1
                break # Salir del bucle de cadenas una vez encontrada
        break # Salir del bucle de modelos una vez encontrado

print(f"\nTotal de átomos listados en la Cadena '{chain_id_to_explore}': {atom_count}")


Listando átomos de la Cadena 'A' en el Modelo 0...
  Cadena encontrada: A
      Átomo: N, Elemento: N, Coordenadas: [5.8371e+01 1.2000e-02 2.3479e+01]
      Átomo: CA, Elemento: C, Coordenadas: [57.582 -1.151 22.974]
      Átomo: C, Elemento: C, Coordenadas: [58.212 -2.477 23.401]
      Átomo: O, Elemento: O, Coordenadas: [57.502 -3.448 23.648]
      Átomo: CB, Elemento: C, Coordenadas: [57.366 -1.093 21.436]
      Átomo: CG1, Elemento: C, Coordenadas: [56.365 -2.164 20.969]
      Átomo: CG2, Elemento: C, Coordenadas: [58.686 -1.227 20.682]
      Átomo: CD1, Elemento: C, Coordenadas: [55.001 -2.104 21.629]
      Átomo: N, Elemento: N, Coordenadas: [59.539 -2.502 23.505]
      Átomo: CA, Elemento: C, Coordenadas: [60.265 -3.707 23.908]
      Átomo: C, Elemento: C, Coordenadas: [59.847 -4.176 25.305]
      Átomo: O, Elemento: O, Coordenadas: [59.727 -5.379 25.553]
      Átomo: CB, Elemento: C, Coordenadas: [61.776 -3.476 23.849]
      Átomo: CG, Elemento: C, Coordenadas: [62.282 -3.048 2

### 4. Plantilla para calcular la distancia euclidiana entre dos átomos

Modifica los valores de `residue_id_1`, `atom_name_1`, `residue_id_2` y `atom_name_2` en la siguiente celda para calcular la distancia entre diferentes átomos.

In [ ]:
from Bio.PDB import PDBParser
import numpy as np # Importamos numpy para el cálculo de la norma

# --- PARÁMETROS CONFIGURABLES ---
# Modifica estos valores para especificar los átomos que quieres analizar
model_to_use = 0    # ID del modelo (generalmente 0 si solo hay uno)
chain_to_use = 'A'  # ID de la cadena (ej: 'A', 'B', etc.)

# Primer átomo
residue_id_1 = 2    # ID numérico del residuo
atom_name_1 = 'CA'  # Nombre del átomo (ej: 'N', 'CA', 'C', 'O', 'CB', etc.)

# Segundo átomo
residue_id_2 = 6    # ID numérico del residuo
atom_name_2 = 'CA'  # Nombre del átomo
# ---------------------------------

# Asegurarse de que la estructura PDB esté cargada.
# Si 'structure' no está disponible en este momento, descomenta y ejecuta lo siguiente:
# parser = PDBParser()
# pdb_file_path = './5DM2.pdb' # Asegúrate de que esta ruta sea correcta
# structure = parser.get_structure('5DM2', pdb_file_path)

atom1_obj = None
atom2_obj = None

# Buscar los átomos específicos en la estructura
for model in structure:
    if model.get_id() == model_to_use:
        for chain in model:
            if chain.get_id() == chain_to_use:
                for residue in chain:
                    res_id_tuple = residue.get_id()
                    # Biopython almacena el ID del residuo en el segundo elemento de la tupla (residuo_numero)
                    if res_id_tuple[1] == residue_id_1:
                        if atom_name_1 in residue:
                            atom1_obj = residue[atom_name_1]
                    if res_id_tuple[1] == residue_id_2:
                        if atom_name_2 in residue:
                            atom2_obj = residue[atom_name_2]
                break # Salir del bucle de cadenas una vez encontrada
        break # Salir del bucle de modelos una vez encontrado

if atom1_obj is not None and atom2_obj is not None:
    # Obtener los vectores de posición como arrays de NumPy
    vec1 = atom1_obj.get_vector().get_array()
    vec2 = atom2_obj.get_vector().get_array()

    # Calcular la distancia euclidiana usando numpy.linalg.norm
    distance = np.linalg.norm(vec1 - vec2)
    print(f"La distancia euclidiana entre el átomo {atom_name_1} (Residuo {residue_id_1}) y el átomo {atom_name_2} (Residuo {residue_id_2}) en la cadena {chain_to_use} es: {distance:.3f} Å")
else:
    print(f"Advertencia: No se pudieron encontrar ambos átomos en la cadena {chain_to_use} del modelo {model_to_use}.")
    if atom1_obj is None:
        print(f"  - El átomo {atom_name_1} del residuo {residue_id_1} no fue encontrado.")
    if atom2_obj is None:
        print(f"  - El átomo {atom_name_2} del residuo {residue_id_2} no fue encontrado.")

La distancia euclidiana entre el átomo CA (Residuo 2) y el átomo CA (Residuo 6) en la cadena A es: 6.178 Å


preparan

# Task
Analyze structural data of relevant dust mite allergens by identifying their PDB IDs, downloading the corresponding PDB files, performing basic structural analysis (models, chains, residues, atoms), identifying surface-exposed residues and potential epitopic regions, and calculating Euclidean distances between key residues within these epitopic regions to inform the development of allergy solutions such as inhibitors or epitope-based vaccines.

## Identificar Alérgenos Relevantes y Sus PDB IDs

### Subtask:
Identificar alérgenos de ácaros del polvo o componentes del polvo relevantes y obtener sus IDs de PDB para el análisis estructural.


## Identificar Alérgenos Relevantes y Sus PDB IDs

### Subtask:
Identificar alérgenos de ácaros del polvo o componentes del polvo relevantes y obtener sus IDs de PDB para el análisis estructural.

#### Instructions
1.  **Realizar Búsqueda Manual (fuera de este cuaderno):** Utiliza bases de datos en línea como RCSB PDB (https://www.rcsb.org/) o AllergenOnline (https://www.allergenonline.org/) para buscar alérgenos relevantes de ácaros del polvo (por ejemplo, Der p 1, Der f 1, Der p 2, etc.).
2.  **Identificar PDB IDs:** Para cada alérgeno relevante que encuentres, anota su código PDB de 4 caracteres (ej. '5DM2').
3.  **Crear Lista de PDB IDs:** En una nueva celda de código, define una lista de Python llamada `allergen_pdb_ids` que contenga los PDB IDs que has identificado. Por ejemplo: `allergen_pdb_ids = ['5DM2', '1A8M', 'otra_id', ...]`. Esta lista se utilizará en los siguientes pasos para descargar y analizar los archivos.


**Reasoning**:
The previous markdown block outlined the instructions for creating a Python list of PDB IDs. This code block provides an initial placeholder list for the user to populate with the PDB IDs they identified manually.



In [ ]:
# Esta lista contendrá los IDs de PDB de los alérgenos que identificaste manualmente.
# Ejemplo: allergen_pdb_ids = ['5DM2', '1A8M', '4YQ1']
allergen_pdb_ids = ['5DM2'] # Comienza con un ejemplo; el usuario debe modificarlo.

print(f"Lista de IDs de PDB para alérgenos: {allergen_pdb_ids}")

Lista de IDs de PDB para alérgenos: ['5DM2']


## Descargar Múltiples Archivos PDB de Alérgenos

### Subtask:
Descargar todos los archivos PDB correspondientes a los IDs de alérgenos identificados en la lista `allergen_pdb_ids`.


**Reasoning**:
The user wants to download multiple PDB files based on the `allergen_pdb_ids` list. I will iterate through this list and call the previously defined `download_pdb_file` function for each ID.



In [ ]:
for pdb_id in allergen_pdb_ids:
    download_pdb_file(pdb_id)
print("Todos los archivos PDB de alérgenos han sido descargados.")

Descargando 5DM2 de https://files.rcsb.org/download/5DM2.pdb...
Archivo ./5DM2.pdb descargado exitosamente.
Todos los archivos PDB de alérgenos han sido descargados.


## Análisis Estructural Básico de Alérgenos

### Subtask:
Parsear las estructuras PDB descargadas, extrayendo información sobre modelos, cadenas, residuos y átomos para entender la composición general de cada alérgeno.


**Reasoning**:
The user wants to parse each downloaded PDB file and extract basic structural information such as models, chains, residues, and atoms. I will write a Python code block to iterate through the `allergen_pdb_ids` list, parse each PDB file, and print the requested details.



In [ ]:
from Bio.PDB import PDBParser

# Crear un objeto PDBParser si no existe
# Aseguramos que 'parser' esté definido incluso si las celdas anteriores no se ejecutaron en orden.
# Aunque ya se importó y creó en celdas anteriores, es buena práctica asegurarse para esta tarea específica.
parser = PDBParser()

print("Iniciando análisis estructural básico para los alérgenos...\n")

for pdb_id in allergen_pdb_ids:
    pdb_file_path = f'./{pdb_id}.pdb'
    try:
        structure = parser.get_structure(pdb_id, pdb_file_path)

        print(f"=== Análisis para la estructura PDB: {structure.get_id()} ===")
        print(f"ID de la estructura: {structure.get_id()}")
        print(f"Número de modelos: {len(structure)}")

        for model in structure:
            print(f"  Modelo ID: {model.get_id()}")
            print(f"  Número de cadenas en el modelo {model.get_id()}: {len(model)}")
            for chain in model:
                print(f"    Cadena ID: {chain.get_id()}")
                print(f"    Número de residuos en la cadena {chain.get_id()}: {len(chain)}")
                print("      Primeros 5 residuos (Residuo: Nombre, ID):")
                # Mostrar los primeros 5 residuos como ejemplo para evitar una salida muy larga
                for i, residue in enumerate(chain):
                    if i < 5:
                        # get_resname() para el nombre, get_id()[1] para el número de secuencia
                        print(f"        Residuo: {residue.get_resname()}, ID: {residue.get_id()[1]}")
                    else:
                        break
                if len(chain) > 5:
                    print("        ...")
        print("\n")

    except Exception as e:
        print(f"Error al procesar el archivo {pdb_file_path}: {e}\n")

print("Análisis estructural básico completado para todos los alérgenos.\n")

Iniciando análisis estructural básico para los alérgenos...

=== Análisis para la estructura PDB: 5DM2 ===
ID de la estructura: 5DM2
Número de modelos: 1
  Modelo ID: 0
  Número de cadenas en el modelo 0: 1
    Cadena ID: A
    Número de residuos en la cadena A: 497
      Primeros 5 residuos (Residuo: Nombre, ID):
        Residuo: ILE, ID: 2
        Residuo: GLN, ID: 3
        Residuo: GLU, ID: 4
        Residuo: LYS, ID: 5
        Residuo: ILE, ID: 6
        ...


Análisis estructural básico completado para todos los alérgenos.

